# Análisis exploratorio — BaselineTrend

Este cuaderno es para **mirar los datos con tus propios ojos**, no para tomar
decisiones. Las decisiones salen de los reportes reproducibles (`tools/report.py`,
`tools/walk_forward.py`), que se ejecutan igual siempre y quedan registrados.

Un cuaderno es interactivo y no reproducible por naturaleza: ejecutas celdas en
distinto orden, cambias un valor, vuelves atrás. Está bien para entender; está
mal para concluir.

> **Cuidado con el sesgo de anticipación aquí.** En un cuaderno tienes el
> dataframe entero, y mirar hacia adelante es trivial y a veces útil (para
> entender qué pasó después de una señal). Nada de lo que hagas aquí debe
> copiarse a la estrategia sin comprobar que solo usa información pasada.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "user_data" / "strategies"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("raiz:", RAIZ)

## 1. Cargar los datos y calcular los indicadores

Se usa **la estrategia real**, no una reimplementación. Si la reimplementaras
aquí, estarías analizando un sistema distinto del que opera.

In [ ]:
from BaselineTrend import BaselineTrend

PAR = "BTC/USDT"
archivo = RAIZ / "user_data" / "data" / f"{PAR.replace('/', '_')}-1h.feather"

df = pd.read_feather(archivo)
df["date"] = pd.to_datetime(df["date"], utc=True)

estrategia = BaselineTrend({
    "stake_currency": "USDT", "max_open_trades": 3, "dry_run": True,
    "timeframe": "1h", "runmode": "backtest", "exchange": {"name": "binance"},
})

df = estrategia.populate_indicators(df, {"pair": PAR})
df = estrategia.populate_entry_trend(df, {"pair": PAR})
df = estrategia.populate_exit_trend(df, {"pair": PAR})

print(f"{len(df):,} velas · {df['date'].min():%Y-%m-%d} → {df['date'].max():%Y-%m-%d}")
print(f"señales de entrada: {int(df.get('enter_long', pd.Series(dtype=int)).fillna(0).sum())}")
print(f"señales de salida : {int(df.get('exit_long', pd.Series(dtype=int)).fillna(0).sum())}")

## 2. ¿Cuál de los cuatro filtros descarta más?

Es la pregunta más informativa que se le puede hacer a esta estrategia. Si un
filtro no descarta nada, sobra: solo añade una variable que ajustar. Si uno
descarta casi todo, la estrategia es ese filtro y lo demás es decoración.

In [ ]:
cruce = (df["ema_rapida"] > df["ema_lenta"]) & ~(df["ema_rapida"] > df["ema_lenta"]).shift(1, fill_value=False)

filtros = {
    "1. cruce alcista EMA20/50": cruce,
    "2. close > EMA200":         df["close"] > df["ema_regimen"],
    "3. RSI entre 40 y 70":      df["rsi"].between(40, 70, inclusive="neither"),
    "4. volumen > SMA(20)":      df["volume"] > df["volumen_sma"],
}

print("Por separado, sobre todas las velas:")
for nombre, cond in filtros.items():
    print(f"  {nombre:<28} {cond.sum():>7,} velas  ({cond.mean():6.1%})")

print("\nAcumulando en orden (cuánto queda tras aplicar cada uno):")
resto = pd.Series(True, index=df.index)
previo = len(df)
for nombre, cond in filtros.items():
    resto &= cond
    quedan = int(resto.sum())
    print(f"  tras {nombre:<28} {quedan:>7,}  (descarta {previo - quedan:,})")
    previo = quedan

## 3. Distribución de resultados por operación

En un sistema seguidor de tendencia esta distribución debe ser **asimétrica**:
muchas pérdidas pequeñas y pocas ganancias grandes. Si es simétrica, el sistema
no está dejando correr las ganadoras y el trailing no está haciendo su trabajo.

In [ ]:
import zipfile, json

carpeta = RAIZ / "user_data" / "backtest_results"
zips = sorted(carpeta.glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)

if zips:
    with zipfile.ZipFile(zips[0]) as z:
        interno = next(n for n in z.namelist()
                       if n.endswith(".json") and "meta" not in n and "config" not in n)
        est = next(iter(json.loads(z.read(interno))["strategy"].values()))
    ops = pd.DataFrame(est["trades"])

    fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
    ejes[0].hist(ops["profit_ratio"] * 100, bins=50, color="#2e86de", alpha=0.8)
    ejes[0].axvline(0, color="#576574", linestyle="--")
    ejes[0].set_title("Resultado por operación (%)")
    ejes[0].set_xlabel("%")

    ejes[1].plot(pd.to_datetime(ops["close_date"]), ops["profit_abs"].cumsum(),
                 color="#2e86de")
    ejes[1].axhline(0, color="#576574", linestyle="--")
    ejes[1].set_title("Equity acumulada (USDT)")
    plt.tight_layout()

    print(f"operaciones: {len(ops)}")
    print(f"media ganadora : {ops.loc[ops.profit_abs > 0, 'profit_ratio'].mean() * 100:+.2f} %")
    print(f"media perdedora: {ops.loc[ops.profit_abs < 0, 'profit_ratio'].mean() * 100:+.2f} %")
    print("\nMotivos de salida:")
    print(ops["exit_reason"].value_counts())
else:
    print("No hay backtests. Corre: python tools/run_backtest.py --window in-sample")

## 4. ¿Por dónde sale el sistema?

La distribución de motivos de salida dice mucho:

- Muchos **stop loss** y pocos **trailing**: el stop inicial de 2 ATR está
  demasiado cerca, o las entradas llegan tarde.
- Muchos **exit_signal** (cruce bajista) y pocos trailing: las tendencias no
  duran lo suficiente para armar el trailing en +1.5 ATR.
- Casi todo **trailing**: el sistema está capturando tendencias. Es lo que se
  busca.

In [ ]:
if zips:
    resumen = ops.groupby("exit_reason").agg(
        operaciones=("profit_abs", "size"),
        beneficio_total=("profit_abs", "sum"),
        media_pct=("profit_ratio", lambda s: s.mean() * 100),
        duracion_h=("trade_duration", lambda s: s.mean() / 60),
    ).sort_values("operaciones", ascending=False)
    display(resumen.round(2))

## 5. Tu turno

Preguntas que merece la pena responder mirando los datos, no adivinando:

1. ¿Las operaciones perdedoras se concentran en algún régimen concreto de
   mercado? (compara con la pendiente de la EMA(200))
2. ¿Hay alguna hora del día o día de la semana claramente peor?
3. ¿Cuánto tiempo pasa entre la señal y el máximo de la operación? Si es muy
   corto, se está entrando tarde.
4. ¿Qué pasaría con un stop de 2.5 ATR en vez de 2? **No lo cambies aquí:**
   anótalo y pruébalo con `walk_forward.py`, una idea cada vez.

> Cualquier cosa que descubras aquí es una **hipótesis**, no un hallazgo. Se
> convierte en hallazgo cuando sobrevive al walk-forward contra la baseline.